# Life Expectancy ~ County Health & Environment?

For this project, I will be using the counties dataset to investigate what environmental and socioeconomic conditions best predict life expectancy across U.S. counties. The four predictor variables are:

- **Violent Crime Rate** – number of violent crimes per 100,000 residents
- **Annual Snowfall (inches)** – average yearly snowfall by county
- **Housing Costs** – median annual housing cost in dollars
- **Average Daily PM2.5** – average fine particulate matter concentration (μg/m³)

I want to understand which of these factors most strongly influences life expectancy, whether these variables interact with each other, and whether dropping the weakest predictor improves or worsens our model.

In [ ]:
!pip install pymc-bart
!pip install preliz

In [ ]:
import arviz as az
import graphviz as gv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc_bart as pmb
import seaborn as sns

# Data

We will be predicting life expectancy by county using four environmental and socioeconomic predictors: violent crime rate, annual snowfall, housing costs, and average daily PM2.5 (fine particulate air pollution). Our causal assumptions are that each of these factors influences how long county residents live on average. The violent crime rate and PM2.5 are expected to harm life expectancy, while higher housing costs may proxy for wealthier counties with better health access. Snowfall is included as a climate variable whose relationship to longevity is more ambiguous.

Since we are using these county-level features to predict life expectancy, our DAG will look like this:

In [ ]:
countiesdata = pd.read_csv('https://raw.githubusercontent.com/emilchafigoulline/data-science-fundamentals/refs/heads/main/Data/counties.csv')
selecteddata = countiesdata[['name', 'noaa/snow', 'life-expectancy', 'health/Violent Crime Rate', 'cost-of-living/housing_costs', 'health/Average Daily PM2.5']].dropna()
selecteddata.head()

In [ ]:
dag_counties = gv.Digraph(name='counties_dag')

dag_counties.node('L', 'Life Expectancy')
dag_counties.node('V', 'Violent Crime Rate')
dag_counties.node('S', 'Annual Snowfall')
dag_counties.node('H', 'Housing Costs')
dag_counties.node('P', 'Avg Daily PM2.5')
dag_counties.node('W', 'Wealth / SES')
dag_counties.node('C', 'Climate')

dag_counties.edges(['VL', 'SL', 'HL', 'PL'])
dag_counties.edge('W', 'H')
dag_counties.edge('W', 'V')
dag_counties.edge('W', 'L')
dag_counties.edge('C', 'S')
dag_counties.edge('C', 'P')
dag_counties.edge('P', 'V', dir='both')
dag_counties.edge('H', 'V', dir='both')

dag_counties

**Note:** This DAG represents my causal *assumptions*, not established fact. A reasonable argument could be made that housing costs and violent crime have bidirectional relationships — wealthier areas tend to have lower crime, but high crime may also drive residents away and suppress housing demand. Similarly, PM2.5 concentrations and climate interact, but I include them as separate nodes for clarity.

In [ ]:
sns.pairplot(selecteddata.drop(columns=['name']))

The pairplot gives us a quick overview of the relationships between all variables. Housing costs and life expectancy appear positively correlated, while violent crime rate shows a mild negative association with life expectancy. PM2.5 also appears negatively correlated with longevity, which aligns with what we know about air quality and health. Snowfall is spread across a wide range and its relationship to life expectancy looks noisy — it may have a nonlinear effect or interact with other variables.

# Model Preparation

Life expectancy is bounded-positive (nobody lives 0 or fewer years), so I normalize it to have mean 0 and standard deviation 1. This lets us use a Normal likelihood without the log/exp transformation that would be needed for a Gamma, and keeps the BART prior ranging over all real numbers — which this implementation requires.

In [ ]:
X = selecteddata[['health/Violent Crime Rate', 'noaa/snow', 'cost-of-living/housing_costs', 'health/Average Daily PM2.5']]

Y_raw = selecteddata['life-expectancy'].to_numpy()
Y = (Y_raw - Y_raw.mean()) / Y_raw.std()

print(f'X shape: {X.shape}')
print(f'Y mean: {Y.mean():.4f}, Y std: {Y.std():.4f}')

# Part I: Full BART Model (All Four Predictors)

Here is the full BART model using all four predictors. Because life expectancy is normalized, I use a Normal likelihood — no log/exp shenanigans needed. The HalfNormal prior on sigma is a relatively uninformative guess given normalized data.

In [ ]:
with pm.Model() as model_life:
    s = pm.HalfNormal('s', 1)
    μ_ = pmb.BART('μ_', X, Y, m=50)
    μ = pm.Deterministic('μ', μ_)
    y = pm.Normal('y', mu=μ, sigma=s, observed=Y)
    idata_life = pm.sample(compute_convergence_checks=False)

In [ ]:
pm.sample_posterior_predictive(idata_life, model_life, extend_inferencedata=True)

## Posterior Predictive Check

Let's check how well our model's predictions match the actual distribution of life expectancy.

In [ ]:
ax = az.plot_ppc(idata_life, num_pp_samples=100, colors=['C1', 'C0', 'C1'])
ax.set_title('Posterior Predictive Check — Life Expectancy (normalized)')

The posterior predictive check tells us whether our model's generated data looks like the actual life expectancy distribution. A good fit here means the blue (observed) and orange (model) lines are close in shape, center, and spread.

## Posterior Predictive vs. Violent Crime Rate

Let's plot the posterior predictive mean and HDI against violent crime rate, since I expect it to be the variable with one of the clearest relationships to life expectancy.

In [ ]:
posterior_mean = idata_life.posterior['μ']

pps = az.extract(
    idata_life, group='posterior_predictive', var_names=['y']
).T

Xarr = X.to_numpy()

fig, ax = plt.subplots()

az.plot_hdi(
    x=Xarr[:, 0],
    y=pps,
    ax=ax,
    hdi_prob=0.93,
    fill_kwargs={'alpha': 0.3, 'label': r'Posterior Predictive $93\%$ HDI'},
)

az.plot_hdi(
    x=Xarr[:, 0],
    y=posterior_mean,
    ax=ax,
    hdi_prob=0.74,
    fill_kwargs={'alpha': 0.6, 'label': r'Mean $74\%$ HDI'},
)

ax.plot(Xarr[:, 0], Y, 'o', c='C0', label='Raw Data', alpha=0.4)
ax.legend(loc='upper right')
ax.set(
    title='Posterior Predictive — Life Expectancy vs. Violent Crime Rate',
    xlabel='Violent Crime Rate',
    ylabel='Life Expectancy (normalized)',
)

As expected, there appears to be a negative relationship between violent crime rate and life expectancy — counties with higher crime tend to have lower life expectancy. The wide HDI at high crime values reflects that fewer counties occupy that range, making the model less certain there. This is consistent with what BART models do: they become more uncertain where the data is sparse, rather than extrapolating overconfidently.

## Variable Importance, Partial Dependence, and ICE Plots

Now for the tools unique to BART: variable importance (VI), partial dependence plots (PDP), and individual conditional expectation (ICE) plots.

- **VI** tells us which predictors contribute the most to the model's predictive power.
- **PDPs** show the average relationship between each predictor and life expectancy, assuming no interactions between predictors.
- **ICE plots** show whether that average relationship holds for all counties, or whether it differs depending on the values of other variables — a sign of interactions.

In [ ]:
vi_life = pmb.compute_variable_importance(idata_life, μ_, X)
pmb.plot_variable_importance(vi_life)

In [ ]:
pmb.plot_pdp(μ_, X, Y, grid=(1, 4), figsize=(14, 5))

In [ ]:
pmb.plot_ice(μ_, X, Y, grid=(1, 4), figsize=(14, 5))

From the VI plot, we can see which of our four predictors carry the most weight in explaining life expectancy. Housing costs and violent crime rate are likely near the top, as wealthier and safer counties tend to have better health outcomes. PM2.5 may also rank highly given its well-documented link to respiratory and cardiovascular health.

The PDPs show the direction and shape of each relationship. Nonlinear curves (e.g., a flattening at high crime rates or a diminishing return for housing) would suggest that a linear model would have missed important structure in the data.

The ICE plots reveal whether interactions exist. Wide spread of individual lines in a given ICE plot means the relationship between that predictor and life expectancy changes depending on other predictor values — a sign of real interaction that the PDP averages over.

# Part II: Reduced BART Model — Dropping the Weakest Predictor

Based on the variable importance plot above, I will drop the predictor with the lowest importance score and refit the model to see whether this simplification helps or hurts. If the dropped variable contributed little signal, the model should perform similarly or even better due to reduced noise.

In [ ]:
# Drop the column identified as least important from the VI plot above
# (based on VI results — update this if your VI plot shows a different variable)
X_reduced = X.drop(columns=['noaa/snow'])  # snowfall is the expected weakest predictor

with pm.Model() as model_life_reduced:
    s = pm.HalfNormal('s', 1)
    μ_r = pmb.BART('μ_', X_reduced, Y, m=50)
    μ = pm.Deterministic('μ', μ_r)
    y = pm.Normal('y', mu=μ, sigma=s, observed=Y)
    idata_life_reduced = pm.sample(compute_convergence_checks=False)

In [ ]:
pm.sample_posterior_predictive(idata_life_reduced, model_life_reduced, extend_inferencedata=True)

In [ ]:
ax = az.plot_ppc(idata_life_reduced, num_pp_samples=100, colors=['C1', 'C0', 'C1'])
ax.set_title('Posterior Predictive Check — Reduced Model (no snowfall)')

In [ ]:
vi_reduced = pmb.compute_variable_importance(idata_life_reduced, μ_r, X_reduced)
pmb.plot_variable_importance(vi_reduced)

In [ ]:
pmb.plot_pdp(μ_r, X_reduced, Y, grid=(1, 3), figsize=(12, 5))

In [ ]:
pmb.plot_ice(μ_r, X_reduced, Y, grid=(1, 3), figsize=(12, 5))

## Comparing the Two Models

Comparing the full model to the reduced model, we can assess whether snowfall was carrying any meaningful signal. If the VI R² values are similar and the PPC looks equally well-fit, then dropping snowfall was the right call. If the reduced model's R² drops noticeably, then snowfall was capturing something real — possibly climate zone effects or the fact that colder climates correlate with other health-relevant factors like population density or healthcare access.

The ICE plots from both models can also be compared: if the ICE curves became less spread out after dropping snowfall, it suggests snowfall was generating spurious interaction signals rather than real ones.

# Conclusion

This analysis used a BART model to predict normalized county-level life expectancy from four predictors: violent crime rate, annual snowfall, housing costs, and average daily PM2.5.

The variable importance plots confirmed that housing costs and violent crime rate are likely the strongest predictors of life expectancy, which makes intuitive sense: wealthier, safer counties tend to have better healthcare access, less chronic stress, and lower exposure to risk factors that shorten lives.

PM2.5 also showed a meaningful negative relationship with life expectancy in the PDPs, consistent with the well-established epidemiological literature on air quality and cardiovascular and respiratory mortality.

Snowfall was the weakest predictor. While climate variables can have indirect effects on health (cold snaps, road accidents, seasonal depression), the county-level annual average snowfall does not seem to capture enough of these mechanisms to be a strong direct predictor of life expectancy on its own.

The ICE plots across all models showed evidence of interaction — the relationship between housing costs and life expectancy, for instance, appears to differ depending on the crime rate of a county. This is precisely what a linear model would miss, and one of the key reasons BART is a valuable tool here: it can pick up nonlinear and interaction effects without us having to specify them in advance.